In [ ]:
import os
import random
import pickle
import numpy as np
import tensorflow as tf
from keras.models import Sequential
from keras.layers import (Conv1D, MaxPooling1D, BatchNormalization,
                          Activation, Flatten, Dense, Dropout)
from keras.layers import ELU
from keras.initializers import glorot_uniform
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import Callback
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_curve, classification_report
from natsort import natsorted
import glob
import zipfile
import shutil
import csv
from tqdm import tqdm
 
# ─────────────────────────────────────────────
# REPRODUCIBILITY
# ─────────────────────────────────────────────
random.seed(0)
np.random.seed(0)
tf.random.set_seed(0)
 
 
# ═════════════════════════════════════════════════════════════════════════════
# CONFIGURATION  ← change these as needed
# ═════════════════════════════════════════════════════════════════════════════
CLOSED_INPUT_PATH   = ""   # folder / zip containing closed .txt files
OPEN_INPUT_PATH     = ""     # folder / zip containing open .txt files
 
WORKING_DIR         = "/kaggle/working"
 
# Feature extraction
MAX_MATRIX_LEN      = 2000
MAX_TIME            = 60
 
# Open-world sample budget  ← how many open samples to use for train / test
OPEN_TRAIN_SAMPLES  = 400    # drawn from shuffled open corpus
OPEN_TEST_SAMPLES   = 1000
 
# Training
NB_EPOCH            = 30
BATCH_SIZE          = 32
LEARNING_RATE       = 0.002
 
# Output CSV
OUTPUT_CSV          = os.path.join(WORKING_DIR, "precision_recall.csv")

In [ ]:
def winlap(instance, max_matrix_len=MAX_MATRIX_LEN, max_time=MAX_TIME):
    """Overlapping time-amplitude matrix features."""
    times, sizes = [], []
    for line in instance:
        try:
            t, s = line.split()
            times.append(float(t))
            sizes.append(int(float(s.split('\n')[0])))
        except Exception:
            continue
 
    n = max_matrix_len * 2
    feature = [0] * n * 2
    for i in range(len(sizes)):
        idx = int(times[i] / max_time * n)
        idx = min(idx, n - 1)
        feature[idx]     += 1
        feature[idx + n] += -sizes[i]
        pre = max(idx - 1, 0)
        feature[pre]     += 1
        feature[pre + n] += -sizes[i]
    return feature
 
 
def collect_txt_files(input_path, staging_dir):
    """Return list of .txt files; extracts zips if present."""
    if os.path.exists(staging_dir):
        shutil.rmtree(staging_dir)
    os.makedirs(staging_dir, exist_ok=True)
 
    zip_files = glob.glob(f"{input_path}/**/*.zip", recursive=True)
    if zip_files:
        for zf in zip_files:
            with zipfile.ZipFile(zf, 'r') as z:
                z.extractall(staging_dir)
        return glob.glob(f"{staging_dir}/**/*.txt", recursive=True)
 
    txt = glob.glob(f"{input_path}/**/*.txt", recursive=True)
    return txt
 
 
def extract_features(files):
    """Extract features; label = everything before the last '_' in filename."""
    X, y = [], []
    for f in tqdm(files, desc="  features"):
        try:
            with open(f, 'r') as fh:
                lines = fh.readlines()
            label = os.path.basename(f).rsplit('_', 1)[0]
            X.append(winlap(lines))
            y.append(label)
        except Exception as e:
            print(f"  [skip] {f}: {e}")
    return np.array(X, dtype='float32'), np.array(y)
 

In [ ]:
class DKFNet:
    @staticmethod
    def build(input_len, num_classes):
        filters     = [None, 32,  64,  128, 256,  512]
        kernel      = [None,  8,   8,    8,   8,    8]
        conv_stride = [None,  1,   1,    1,   1,    1]
        pool_stride = [None,  4,   4,    4,   4,    4]
        pool_sz     = [None,  8,   8,    8,   8,    8]
 
        m = Sequential()
 
        # Block 1 – ELU activations
        m.add(Conv1D(filters[1], kernel[1], input_shape=(input_len, 1),
                     strides=conv_stride[1], padding='same', name='b1_conv1'))
        m.add(BatchNormalization())
        m.add(ELU(alpha=1.0, name='b1_act1'))
        m.add(Conv1D(filters[1], kernel[1], strides=conv_stride[1],
                     padding='same', name='b1_conv2'))
        m.add(BatchNormalization())
        m.add(ELU(alpha=1.0, name='b1_act2'))
        m.add(MaxPooling1D(pool_sz[1], strides=pool_stride[1], padding='same'))
        m.add(Dropout(0.2))
 
        # Blocks 2-5 – ReLU activations
        for blk in range(2, 6):
            m.add(Conv1D(filters[blk], kernel[blk], strides=conv_stride[blk],
                         padding='same', name=f'b{blk}_conv1'))
            m.add(BatchNormalization())
            m.add(Activation('relu', name=f'b{blk}_act1'))
            m.add(Conv1D(filters[blk], kernel[blk], strides=conv_stride[blk],
                         padding='same', name=f'b{blk}_conv2'))
            m.add(BatchNormalization())
            m.add(Activation('relu', name=f'b{blk}_act2'))
            m.add(MaxPooling1D(pool_sz[blk], strides=pool_stride[blk], padding='same'))
            m.add(Dropout(0.2))
 
        # FC head
        m.add(Flatten())
        m.add(Dense(512, kernel_initializer=glorot_uniform(seed=0), name='fc1'))
        m.add(BatchNormalization())
        m.add(Activation('relu'))
        m.add(Dropout(0.7))
 
        m.add(Dense(512, kernel_initializer=glorot_uniform(seed=0), name='fc2'))
        m.add(BatchNormalization())
        m.add(Activation('relu'))
        m.add(Dropout(0.5))
 
        m.add(Dense(num_classes, kernel_initializer=glorot_uniform(seed=0), name='fc3'))
        m.add(Activation('softmax'))
        return m
 
 
class EpochLogger(Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"  Epoch {epoch+1:3d}  "
              f"loss={logs['loss']:.4f}  acc={logs['accuracy']:.4f}  "
              f"val_loss={logs['val_loss']:.4f}  val_acc={logs['val_accuracy']:.4f}")
 

In [ ]:
def main():
    os.makedirs(WORKING_DIR, exist_ok=True)
 
    # ── 1. Load closed dataset ───────────────────────────────────────────────
    print("\n[1/6] Loading CLOSED dataset ...")
    closed_files = natsorted(collect_txt_files(CLOSED_INPUT_PATH, "/tmp/closed_staging"))
    if not closed_files:
        raise FileNotFoundError(f"No .txt files found in {CLOSED_INPUT_PATH}")
    print(f"  Found {len(closed_files)} files")
 
    X_closed, y_closed_raw = extract_features(closed_files)
 
    le = LabelEncoder()
    y_closed = le.fit_transform(y_closed_raw)          # integers 0..N-1
    N = len(le.classes_)
    FEATURE_LEN   = X_closed.shape[1]                  # 8000
    TOTAL_CLASSES = N + 1                              # N closed + 1 open-world
 
    print(f"  Closed classes (N)  : {N}")
    print(f"  Feature length      : {FEATURE_LEN}")
    print(f"  Total model classes : {TOTAL_CLASSES}  (N+1)")
 
    # ── 2. Load open dataset ─────────────────────────────────────────────────
    print("\n[2/6] Loading OPEN dataset ...")
    open_files = natsorted(collect_txt_files(OPEN_INPUT_PATH, "/tmp/open_staging"))
    if not open_files:
        raise FileNotFoundError(f"No .txt files found in {OPEN_INPUT_PATH}")
    print(f"  Found {len(open_files)} files")
 
    X_open_all, _ = extract_features(open_files)
 
    # Shuffle all open samples once – no per-label balance needed
    rng = np.random.default_rng(seed=0)
    idx_all = rng.permutation(len(X_open_all))
    X_open_all = X_open_all[idx_all]
 
    needed = OPEN_TRAIN_SAMPLES + OPEN_TEST_SAMPLES
    if len(X_open_all) < needed:
        raise ValueError(
            f"Not enough open samples: have {len(X_open_all)}, need {needed}")
 
    X_open_train = X_open_all[:OPEN_TRAIN_SAMPLES]
    X_open_test  = X_open_all[OPEN_TRAIN_SAMPLES : OPEN_TRAIN_SAMPLES + OPEN_TEST_SAMPLES]
    y_open_train = np.full(OPEN_TRAIN_SAMPLES, N, dtype=int)   # label = N
    y_open_test  = np.full(OPEN_TEST_SAMPLES,  N, dtype=int)
 
    print(f"  Open train samples  : {OPEN_TRAIN_SAMPLES}")
    print(f"  Open test  samples  : {OPEN_TEST_SAMPLES}")
 
    # ── 3. Build train / val / test splits ───────────────────────────────────
    print("\n[3/6] Splitting data ...")
 
    # Closed: stratified 70 / 15 / 15
    X_cl_tr, X_cl_tmp, y_cl_tr, y_cl_tmp = train_test_split(
        X_closed, y_closed, test_size=0.30, random_state=42, stratify=y_closed)
    X_cl_val, X_cl_te, y_cl_val, y_cl_te = train_test_split(
        X_cl_tmp, y_cl_tmp, test_size=0.50, random_state=42, stratify=y_cl_tmp)
 
    # Open: 70% train, 30% val from OPEN_TRAIN_SAMPLES; OPEN_TEST_SAMPLES for test
    split = int(OPEN_TRAIN_SAMPLES * 0.70)
    X_op_tr,  y_op_tr  = X_open_train[:split],  y_open_train[:split]
    X_op_val, y_op_val = X_open_train[split:],   y_open_train[split:]
    X_op_te,  y_op_te  = X_open_test,            y_open_test
 
    # Combine closed + open
    X_train = np.concatenate([X_cl_tr,  X_op_tr])
    y_train = np.concatenate([y_cl_tr,  y_op_tr])
    X_val   = np.concatenate([X_cl_val, X_op_val])
    y_val   = np.concatenate([y_cl_val, y_op_val])
    X_test  = np.concatenate([X_cl_te,  X_op_te])
    y_test  = np.concatenate([y_cl_te,  y_op_te])
 
    # Shuffle each split in-place
    for Xs, ys in [(X_train, y_train), (X_val, y_val), (X_test, y_test)]:
        p = np.random.permutation(len(Xs))
        Xs[:], ys[:] = Xs[p], ys[p]
 
    print(f"  Train : {X_train.shape}   Val : {X_val.shape}   Test : {X_test.shape}")
 
    # Add channel dim → (samples, length, 1)
    X_train = X_train[:, :, np.newaxis]
    X_val   = X_val[:,   :, np.newaxis]
    X_test  = X_test[:,  :, np.newaxis]
 
    y_train_oh = to_categorical(y_train, TOTAL_CLASSES)
    y_val_oh   = to_categorical(y_val,   TOTAL_CLASSES)
    y_test_oh  = to_categorical(y_test,  TOTAL_CLASSES)
 
    # ── 4. Build & train model ───────────────────────────────────────────────
    print("\n[4/6] Building model ...")
    model = DKFNet.build(input_len=FEATURE_LEN, num_classes=TOTAL_CLASSES)
    model.compile(
        loss="categorical_crossentropy",
        optimizer=Adamax(learning_rate=LEARNING_RATE,
                         beta_1=0.9, beta_2=0.999, epsilon=1e-8),
        metrics=["accuracy"]
    )
    model.summary()
 
    print(f"\n[5/6] Training for {NB_EPOCH} epochs ...")
    model.fit(
        X_train, y_train_oh,
        batch_size=BATCH_SIZE,
        epochs=NB_EPOCH,
        verbose=0,
        validation_data=(X_val, y_val_oh),
        callbacks=[EpochLogger()]
    )
 
    # Save artefacts
    model.save(os.path.join(WORKING_DIR, "openworld_model.keras"))
    with open(os.path.join(WORKING_DIR, "label_encoder.pkl"), 'wb') as f:
        pickle.dump(le, f)
 
    # ── 5. Evaluate ──────────────────────────────────────────────────────────
    print("\n[6/6] Evaluating ...")
 
    y_prob = model.predict(X_test, verbose=0)           # (n_test, N+1)
    y_pred = np.argmax(y_prob, axis=1)
 
    # ── 5a. Classification report for all N+1 classes ────────────────────────
    # Build human-readable label names: closed labels + "open-world"
    class_names = list(le.classes_) + ["open-world"]   # length = N+1
    
    print("\n" + "=" * 60)
    print("CLASSIFICATION REPORT  (N+1 classes)")
    print("=" * 60)
    
    report = classification_report(
        y_test,
        y_pred,
        labels=list(range(TOTAL_CLASSES)),
        target_names=class_names,
        zero_division=0,
        digits=4   # <-- lấy 4 chữ số sau dấu phẩy
    )
    
    print(report)
 
    # ── 5b. Precision / recall curve (open-world class) ──────────────────────
    # Binary: 1 = open-world sample, 0 = closed sample
    y_true_bin      = (y_test == N).astype(int)
    open_class_prob = y_prob[:, N]
 
    precision_arr, recall_arr, _ = precision_recall_curve(y_true_bin, open_class_prob)
 
    with open(OUTPUT_CSV, 'w', newline='') as csvf:
        writer = csv.writer(csvf)
        writer.writerow(["precision", "recall"])
        for p, r in zip(precision_arr, recall_arr):
            writer.writerow([p, r])
 
    print(f"Precision-recall curve → {len(precision_arr)} points saved to {OUTPUT_CSV}")

# ── 5c. ROC curve (open-world class) ─────────────────────────────────────
    from sklearn.metrics import roc_curve, auc as sk_auc

    fpr, tpr, roc_thresh = roc_curve(y_true_bin, open_class_prob)
    roc_auc = sk_auc(fpr, tpr)

    ROC_CSV = os.path.join(WORKING_DIR, "roc.csv")
    with open(ROC_CSV, 'w', newline='') as csvf:
        writer = csv.writer(csvf)
        writer.writerow(["fpr", "tpr", "threshold"])
        for f, t, th in zip(fpr, tpr, roc_thresh):
            writer.writerow([f, t, th])

    print(f"ROC curve (AUC={roc_auc:.4f}) → {len(fpr)} points saved to {ROC_CSV}")
    
    # ── 5c. Overall test metrics ──────────────────────────────────────────────
    score = model.evaluate(X_test, y_test_oh, verbose=0)
    print(f"\nTest loss     : {score[0]:.4f}")
    print(f"Test accuracy : {score[1]*100:.2f}%")
    print("\nDone ✓")
 
 
if __name__ == "__main__":
    main()
 